# PC-FNO3D réel (Grand Tunis) — diagnostic sur la validation (Colab, CPU)

Avant toute relance de l'entraînement (plateau depuis l'époque 11) : sur les 64 instants de
validation, R² et RMSE par niveau d'**ERA5 interpolé**, du **vent moyen du domaine** (oracle) et
du **modèle** (`best.pt`) ; alignement entrée ERA5 / cible ; part de la variance de la cible due
aux perturbations aléatoires du générateur (instants régénérés sans perturbation).

**Environnement : CPU, mémoire RAM élevée.** Résultats : `MyDrive/pcfno3d_grand_tunis/diagnostic/`.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

## 2. Instants de validation (partie 2)

In [ ]:
# validation seulement : index de la partie 2 (train 2/2 + val) et fichiers des instants de val
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
CACHE.mkdir(parents=True, exist_ok=True)
JEU.mkdir(parents=True, exist_ok=True)
if (GT / "diag_grand_tunis.py").exists():           # script depose a part (plus recent que le zip)
    shutil.copy2(GT / "diag_grand_tunis.py", CODE / "scripts" / "diag_grand_tunis.py")
with zipfile.ZipFile(GT / "jeu.zip") as zf:
    zf.extractall(JEU)
import pandas as pd
with zipfile.ZipFile(GT / "cache_partie_2.zip") as zf:
    with zf.open("index_partie_2.csv") as f:
        idx = pd.read_csv(f)
    idx.to_csv(CACHE / "index.csv", index=False)
    val = set(idx.loc[idx["split"] == "val", "stem"])
    noms = [n for n in zf.namelist() if n.rsplit("_", 1)[0] in val]
    zf.extractall(CACHE, members=noms)
print(f"{len(val)} instants de validation, {len(noms)} fichiers")

## 3. Diagnostic

In [ ]:
SORTIE = Path("/content/diag")
CKPT = Path("/content/best.pt")                     # copie locale : poids mappes en memoire
shutil.copy2(GT / "runs" / "grand_tunis" / "best.pt", CKPT)
lancer("diagnostic", f"python -u scripts/diag_grand_tunis.py -c grand_tunis_colab "
                     f"--donnees {DON / 'grand_tunis'} --ckpt {CKPT} --sortie {SORTIE}")
D = GT / "diagnostic"
D.mkdir(exist_ok=True)
for f in ("diagnostic.json", "alignement_par_instant.json", "r2_rmse_par_niveau.csv"):
    shutil.copy2(SORTIE / f, D / f)
print(sorted(p.name for p in D.iterdir()))